<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 - Cuadros de Mando (Dashboards)

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** Principios de diseño, KPIs, construcción de dashboards

---

## Objetivos
- Comprender los principios de diseño de cuadros de mando (Stephen Few)
- Crear KPIs con contexto y visualización
- Construir un dashboard completo con Plotly

## Principios de Stephen Few

1. **Un vistazo debe ser suficiente** - Información crítica en segundos
2. **Simplicidad sobre complejidad** - Máximo 5-7 visualizaciones
3. **Contexto es imprescindible** - Comparativas, objetivos, tendencias
4. **Jerarquía visual** - Lo más importante arriba-izquierda
5. **Diseñar para la audiencia** - Ejecutivos vs Analistas

## 1. Crear KPIs con contexto

In [ ]:
import plotly.graph_objects as go

def crear_kpi(valor, titulo, objetivo=None, anterior=None):
    """Crea un indicador KPI con contexto"""
    
    fig = go.Figure()
    
    # Número principal con delta
    fig.add_trace(go.Indicator(
        mode="number+delta",
        value=valor,
        title={"text": titulo},
        delta={'reference': anterior, 'relative': True, 'position': "bottom"} if anterior else None,
        domain={'x': [0, 1], 'y': [0.3, 1]}
    ))
    
    # Barra de progreso hacia objetivo
    if objetivo:
        progreso = min(valor / objetivo * 100, 100)
        fig.add_trace(go.Indicator(
            mode="gauge",
            value=progreso,
            gauge={
                'axis': {'range': [0, 100], 'visible': False},
                'bar': {'color': "darkblue"},
                'bgcolor': "lightgray",
                'threshold': {
                    'line': {'color': "red", 'width': 2},
                    'thickness': 0.75,
                    'value': 100
                }
            },
            domain={'x': [0.1, 0.9], 'y': [0, 0.25]}
        ))
    
    fig.update_layout(height=200, margin=dict(t=50, b=20, l=20, r=20))
    return fig

# Ejemplo
fig = crear_kpi(125000, "Ventas Totales (€)", objetivo=150000, anterior=108000)
fig.show()

## 2. Dashboard completo de ventas

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Generar datos de ejemplo
np.random.seed(42)
fechas = pd.date_range('2024-01-01', '2024-12-31', freq='D')
n = len(fechas)

df = pd.DataFrame({
    'fecha': fechas,
    'ventas': np.random.randint(5000, 15000, n) + np.sin(np.arange(n) * 2 * np.pi / 365) * 3000,
    'pedidos': np.random.randint(50, 150, n),
    'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], n),
    'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar', 'Alimentación'], n)
})

df['mes'] = df['fecha'].dt.to_period('M').astype(str)
df['ticket_medio'] = df['ventas'] / df['pedidos']

print(f"Datos generados: {len(df)} registros")
print(df.head())

In [ ]:
# ========================================
# CREAR DASHBOARD COMPLETO
# ========================================

# Calcular KPIs
ventas_total = df['ventas'].sum()
pedidos_total = df['pedidos'].sum()
ticket_medio = ventas_total / pedidos_total
objetivo_ventas = 4000000

# Crear figura con subplots
fig = make_subplots(
    rows=3, cols=4,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"type": "scatter", "colspan": 2}, None, {"type": "bar", "colspan": 2}, None],
        [{"type": "pie"}, {"type": "bar", "colspan": 2}, None, {"type": "table"}]
    ],
    subplot_titles=('', '', '', '', 'Evolución Mensual', 'Ventas por Región',
                    'Por Categoría', 'Top Días', ''),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# Fila 1: KPIs
fig.add_trace(go.Indicator(
    mode="number",
    value=ventas_total,
    title={"text": "Ventas Totales"},
    number={'prefix': "€", 'valueformat': ",.0f"}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=ventas_total,
    delta={'reference': objetivo_ventas, 'relative': True},
    title={"text": "vs Objetivo"}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=pedidos_total,
    title={"text": "Total Pedidos"},
    number={'valueformat': ","}
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode="number",
    value=ticket_medio,
    title={"text": "Ticket Medio"},
    number={'prefix': "€", 'valueformat': ".2f"}
), row=1, col=4)

# Fila 2: Gráficos principales
ventas_mes = df.groupby('mes')['ventas'].sum().reset_index()
fig.add_trace(go.Scatter(
    x=ventas_mes['mes'], y=ventas_mes['ventas'],
    mode='lines+markers', name='Ventas',
    line=dict(color='#1f77b4', width=3)
), row=2, col=1)

ventas_region = df.groupby('region')['ventas'].sum().reset_index()
fig.add_trace(go.Bar(
    x=ventas_region['region'], y=ventas_region['ventas'],
    marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
    name='Región'
), row=2, col=3)

# Fila 3: Detalle
ventas_cat = df.groupby('categoria')['ventas'].sum()
fig.add_trace(go.Pie(
    labels=ventas_cat.index, values=ventas_cat.values,
    hole=0.4
), row=3, col=1)

top_dias = df.nlargest(5, 'ventas')[['fecha', 'ventas']]
fig.add_trace(go.Bar(
    x=top_dias['fecha'].astype(str), y=top_dias['ventas'],
    marker_color='#2ca02c'
), row=3, col=2)

# Configurar layout
fig.update_layout(
    title={
        'text': "📊 DASHBOARD DE VENTAS 2024",
        'x': 0.5,
        'font': {'size': 24}
    },
    height=800,
    showlegend=False,
    template='plotly_white'
)

fig.show()

---

## Recursos adicionales

- [Stephen Few - Dashboard Design](https://www.perceptualedge.com/)
- [Dash by Plotly](https://dash.plotly.com/)
- [Streamlit](https://streamlit.io/)